In [4]:
import pandas as pd 
import torch
from transformers import pipeline
from tqdm.auto import tqdm

In [5]:
# lyu basic 
lyu_basic_prompt_results = pd.read_parquet("/share/pierson/matt/mllmsci/outputs/vqa/bayflood_926k_10pct_lyu_basic_20260112_151140.parquet")

In [6]:
lyu_basic_prompt_results.head()

,sample_id,prompt,image_path,image_url,image_base64,answer,model_response
0,nexar_53afeec954a80d924012432cb70c1081,Analyze and estimate the deepest floodwater le...,/share/ju/nexar_data/2023/2023-09-29/604222325...,None,None,"Based on the image, the floodwater level can b...","Based on the image, the floodwater level can b..."
1,nexar_66384e02c4b2ceda1b331f44ee45e950,Analyze and estimate the deepest floodwater le...,/share/ju/nexar_data/2023/2023-09-29/604222325...,None,None,"Based on the image, the floodwater level can b...","Based on the image, the floodwater level can b..."
2,nexar_85283d2860cd54b22ccef4a210602830,Analyze and estimate the deepest floodwater le...,/share/ju/nexar_data/2023/2023-09-29/604222325...,None,None,The floodwater level in the image can be estim...,The floodwater level in the image can be estim...
3,nexar_4bba636e205d574c150ba661ba3d5e35,Analyze and estimate the deepest floodwater le...,/share/ju/nexar_data/2023/2023-09-29/604222325...,None,None,"Based on the image provided, there is no visib...","Based on the image provided, there is no visib..."
4,nexar_9eab3d72765f7764cd4d1f0171cac6ba,Analyze and estimate the deepest floodwater le...,/share/ju/nexar_data/2023/2023-09-29/604222323...,None,None,"Based on the image provided, the floodwater le...","Based on the image provided, the floodwater le..."


In [ ]:
# Initialize the zero-shot classification pipeline (uses NLI under the hood)
device = 0 if torch.cuda.is_available() else -1
classifier = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=device)

candidate_labels = ["yes, flooded", "no, not flooded"]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

  2026-01-15T22:23:09.387172Z ERROR  Python exception updating progress:, error: PyErr { type: <class 'LookupError'>, value: LookupError(<ContextVar name='shell_parent' at 0x7b7a03d0e390>), traceback: Some(<traceback object at 0x7b77fdbe7700>) }, caller: "src/progress_update.rs:313"
    at /home/runner/work/xet-core/xet-core/error_printer/src/lib.rs:28



In [ ]:
def classify_flood_response(df, classifier, labels):
    responses = df['model_response'].tolist()
    results = []
    
    # Process in batches for speed if needed, or just iterate with tqdm
    for i in tqdm(range(0, len(responses), 16)):
        batch = responses[i:i+16]
        # The hypothesis template helps the NLI model understand the context
        batch_results = classifier(batch, labels, hypothesis_template="The answer to whether there is a flood is {}.")
        results.extend(batch_results)
    
    # Extract the top label and score
    df['nli_label'] = [r['labels'][0] for r in results]
    df['nli_score'] = [r['scores'][0] for r in results]
    df['nli_is_flooded'] = df['nli_label'] == "yes, flooded"
    
    return df

# Apply to a sample or full dataframe
lyu_basic_prompt_results = classify_flood_response(lyu_basic_prompt_results.head(100).copy(), classifier, candidate_labels)

In [ ]:
lyu_basic_prompt_results[['model_response', 'nli_label', 'nli_score']].head(10)